In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
#load the dataset
data=pd.read_csv('../artifacts/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


#### Preprocess the data

In [4]:
data.drop(columns=['RowNumber',"CustomerId","Surname"],inplace=True, axis=1)

In [5]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
data.shape

(10000, 11)

In [7]:
data.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [8]:
data["Geography"].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [9]:
data["Gender"].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [10]:
numerical_features=list(data.select_dtypes(exclude='object').columns)
print(numerical_features)

['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


In [11]:
# Exited is our target feature so we will exclude Exited
numerical_features.remove('Exited')
print(numerical_features)

['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


In [12]:
Ohe_feature=['Geography','Gender']

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
train_df,test_df=train_test_split(data,random_state=43,test_size=0.2)

In [15]:
train_df.shape

(8000, 11)

In [16]:
X_train=train_df.drop('Exited',axis=1)
X_test=test_df.drop('Exited',axis=1)
y_train=train_df['Exited']
y_test=test_df['Exited']

In [17]:
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [18]:
preprocessor=ColumnTransformer(
    transformers=[
        ('standardizer',StandardScaler(),numerical_features),
        ('one hot encoding',OneHotEncoder(handle_unknown='ignore'),Ohe_feature),
    ],
    remainder='passthrough'
)

In [19]:
X_train=preprocessor.fit_transform(X_train)

In [20]:
X_test=preprocessor.transform(X_test)

In [21]:
column_names=preprocessor.get_feature_names_out()
print(column_names)

['standardizer__CreditScore' 'standardizer__Age' 'standardizer__Tenure'
 'standardizer__Balance' 'standardizer__NumOfProducts'
 'standardizer__HasCrCard' 'standardizer__IsActiveMember'
 'standardizer__EstimatedSalary' 'one hot encoding__Geography_France'
 'one hot encoding__Geography_Germany' 'one hot encoding__Geography_Spain'
 'one hot encoding__Gender_Female' 'one hot encoding__Gender_Male']


#### gender and geography columns are removed. 2 columns from gender are added and 3 columns from geography are added. So 11-2+5=14 columns after transformation

In [22]:
X_test.shape,y_test.shape

((2000, 13), (2000,))

In [23]:
X_train.shape,y_train.shape

((8000, 13), (8000,))

In [ ]:
## save the preproceesor
filename="../artifacts/classification/preprocessor.pkl"
with open(filename,'wb') as file:
    pickle.dump(preprocessor,file)

### ANN implementation

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [26]:
# Build Model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),# HL1
    Dense(32,activation='relu'),# HL2
    Dense(1,activation='sigmoid'), # Output Layer
])

In [27]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3009 (11.75 KB)
Trainable params: 3009 (11.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
from tensorflow.keras.optimizers import Adam
opt=Adam(learning_rate=0.01)

In [29]:
# compile the model
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=["accuracy"])

In [30]:
# setup the tensor board
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [31]:
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [32]:
# setup up early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [33]:
# train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 2s 4ms/step - loss: 0.3890 - accuracy: 0.8397 - val_loss: 0.3675 - val_accuracy: 0.8545
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3520 - accuracy: 0.8564 - val_loss: 0.3547 - val_accuracy: 0.8575
Epoch 3/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3439 - accuracy: 0.8585 - val_loss: 0.3600 - val_accuracy: 0.8585
Epoch 4/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3375 - accuracy: 0.8602 - val_loss: 0.3514 - val_accuracy: 0.8590
Epoch 5/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3367 - accuracy: 0.8614 - val_loss: 0.3493 - val_accuracy: 0.8590
Epoch 6/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3329 - accuracy: 0.8661 - val_loss: 0.3557 - val_accuracy: 0.8565
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3328 - accuracy: 0.8646 - val_loss: 0.3511 - val_accuracy: 0.85

In [ ]:
model.save('../artifacts/classification/model.h5')

In [35]:
# Load tensorboard Extension
%load_ext tensorboard

In [36]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 11220), started 1 day, 0:46:34 ago. (Use '!kill 11220' to kill it.)